In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from kemaodanh_oai.data import load_pairs
from sklearn.metrics import accuracy_score, f1_score

In [3]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "data" / "train"

df = load_pairs(TRAIN_DIR / "pairs.csv")

split_df = pd.read_csv(
    PROJECT_ROOT / "configs" / "development_split.csv",
    dtype={"pair_id": str}
)

df = df.merge(
    split_df,
    on="pair_id",
    validate="one_to_one"
)

display(df.head())

,pair_id,image_0,image_1,fake_position,split,fold
0,00001,images/00295.jpg,images/01131.jpg,1,development,1
1,00002,images/00684.jpg,images/00888.jpg,0,development,1
2,00003,images/00016.jpg,images/00059.jpg,0,holdout,-1
3,00004,images/01000.jpg,images/00449.jpg,1,development,1
4,00005,images/01117.jpg,images/01089.jpg,1,development,2


In [4]:
dev_df = df[df["split"] == "development"].copy()

print("Development:", len(dev_df))
print(dev_df["fold"].value_counts().sort_index())

Development: 800
fold
0    267
1    267
2    266
Name: count, dtype: int64


In [5]:
current_fold = 0

train_df = dev_df[dev_df["fold"]!=current_fold].copy()
val_df = dev_df[dev_df["fold"] == current_fold].copy()

print("Training:", len(train_df))
print("Validation:", len(val_df))

Training: 533
Validation: 267


In [6]:
train_ids = set(train_df["pair_id"])
val_ids = set(val_df["pair_id"])

assert train_ids.isdisjoint(val_ids), "Data leakage dectected!"

## Baseline cho feature JPEG

In [7]:
def get_pair_sizes(row, train_dir):
    train_dir = Path(train_dir)

    image_0_path = train_dir / row["image_0"]
    image_1_path = train_dir/ row["image_1"]

    size_0 = image_0_path.stat().st_size
    size_1 = image_1_path.stat().st_size

    return size_0, size_1

rows = val_df.head(5)
for _, row in rows.iterrows():
    size_0, size_1 = get_pair_sizes(row, TRAIN_DIR)
    print(f"Pair ID: {row["pair_id"]}, Image 0: {size_0}, Image 1: {size_1}, True label: {row["fake_position"]}")

Pair ID: 00006, Image 0: 49091, Image 1: 85353, True label: 0
Pair ID: 00009, Image 0: 57399, Image 1: 66145, True label: 1
Pair ID: 00013, Image 0: 98806, Image 1: 61245, True label: 1
Pair ID: 00021, Image 0: 82243, Image 1: 59888, True label: 1
Pair ID: 00022, Image 0: 92335, Image 1: 61466, True label: 1


In [8]:
def predict_jpeg_baseline(df, train_dir):
    train_dir = Path(train_dir)
    predictions = []

    for _,row in df.iterrows():
        size_0, size_1 = get_pair_sizes(row, train_dir)

        fake_pred = 0 if size_0 < size_1 else 1

        predictions.append(fake_pred)

    return predictions

In [9]:
y_true = val_df["fake_position"]
y_pred = predict_jpeg_baseline(val_df, TRAIN_DIR)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro-F1:", f1_score(y_true, y_pred, average="macro"))

Accuracy: 0.7378277153558053
Macro-F1: 0.7378240377062057


In [10]:
fold_scores = []

for fold in range(3):
    val_fold = dev_df[dev_df["fold"]==fold]

    y_pred = predict_jpeg_baseline(val_fold, TRAIN_DIR)
    y_true = val_fold["fake_position"]

    macro_f1 = f1_score(y_true, y_pred, average="macro")

    fold_scores.append(macro_f1)

print("Mean Macro-F1:", np.mean(fold_scores))

Mean Macro-F1: 0.7195331715985581


## Feature Engineering

In [11]:
from kemaodanh_oai.features import extract_rgb_features, extract_pair_features, extract_combine_feature

## Create feature matrix

In [12]:
feature_list = []
for _, row in dev_df.iterrows():
    pair_features = extract_pair_features(row, TRAIN_DIR)
    feature_list.append(pair_features)

X = np.stack(feature_list)

y = dev_df["fake_position"].to_numpy()

print("X shape:", X.shape)
print("Y shape:", y.shape)

X shape: (800, 6)
Y shape: (800,)


In [13]:
folds = dev_df["fold"].to_numpy()

train_idx = np.where(folds != 0)[0]
val_idx = np.where(folds == 0)[0]

X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (533, 6)
Validation: (267, 6)
Training: (533, 6)
Validation: (267, 6)


In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=42
    )
)

In [15]:
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average='macro')

print("Fold 0 - Macro F1:", macro_f1)

Fold 0 - Macro F1: 0.6693684566024991


## Logistic Regression - 3-fold cross-validation

In [16]:
from sklearn.base import clone

folds = dev_df["fold"].to_numpy()

lr_scores = []

for fold in range(3):
    train_idx = np.where(folds!=fold)[0]
    val_idx = np.where(folds==fold)[0]

    X_train = X[train_idx]
    X_val = X[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]

    fold_model = clone(model)

    fold_model.fit(X_train, y_train)

    y_pred = fold_model.predict(X_val)

    score = f1_score(y_pred, y_val, average="macro")

    lr_scores.append(score)

    print(f"Fold {fold}: Macro-F1 = {score:.4f}")

print("Mean Macro-F1:", np.mean(lr_scores))
print("STD Macrod-F1:", np.std(lr_scores))
    

Fold 0: Macro-F1 = 0.6694
Fold 1: Macro-F1 = 0.6554
Fold 2: Macro-F1 = 0.6278
Mean Macro-F1: 0.6508555123618419
STD Macrod-F1: 0.017286377570923113


## JPEG + RGB - Logistics Regression - 3-fold cross-validation

### Extract RGB + JPEG features

In [30]:
from kemaodanh_oai.features import extract_combine_feature

rgb_jpeg_features = []

for _, row in dev_df.iterrows():
    rgb_jpeg_feature = extract_combine_feature(row, TRAIN_DIR)
    rgb_jpeg_features.append(rgb_jpeg_feature)

X_combined = np.stack(rgb_jpeg_features)
y = dev_df['fake_position'].to_numpy()

print("Shape of X:", X.shape)
print("Shape of Y:", y.shape)

Shape of X: (800, 7)
Shape of Y: (800,)


### Train models

In [31]:
combined_scores = []

for fold in range(3):
    train_idx = np.where(folds != fold)[0]
    val_idx = np.where(folds==fold)[0]

    X_train = X_combined[train_idx]
    X_val = X_combined[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]


    fold_model = clone(model)

    fold_model.fit(X_train, y_train)

    y_pred = fold_model.predict(X_val)

    score = f1_score(y_val, y_pred, average='macro')
    lr_scores.append(score)

    print(f"Fold {fold}: {score:.4f}")

print("Mean Macro-F1:", np.mean(lr_scores))
print("STD Macro-F1:", np.std(lr_scores))


Fold 0: 0.7528
Fold 1: 0.7300
Fold 2: 0.7251
Mean Macro-F1: 0.7359673011591997
STD Macro-F1: 0.012056435458596998
